In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
print(tf.__version__)

c:\Users\justi\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


2.21.0


In [2]:
mortgage_df = pd.read_csv("2024_NC_mortgage.csv")
mortgage_df.head()

C:\Users\justi\AppData\Local\Temp\ipykernel_27696\562026718.py:1: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  mortgage_df = pd.read_csv("2024_NC_mortgage.csv")


,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,549300LYRWPSYPK6S325,15500,NC,37001.0,3.700102e+10,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,3170.0,13.22,77900.0,159.0,1347.0,1480.0,51.0
1,2024,549300LYRWPSYPK6S325,15500,NC,37001.0,3.700102e+10,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5217.0,38.78,77900.0,106.0,1649.0,2033.0,25.0
2,2024,549300LYRWPSYPK6S325,99999,NC,37079.0,3.707995e+10,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,3451.0,46.65,71500.0,109.0,1312.0,1868.0,38.0
3,2024,549300LYRWPSYPK6S325,48900,NC,37129.0,3.712901e+10,C,FHA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,2875.0,7.97,96300.0,141.0,912.0,1754.0,31.0
4,2024,549300LYRWPSYPK6S325,39580,NC,37183.0,3.718305e+10,C,VA:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,2838.0,15.54,122300.0,143.0,1070.0,1130.0,41.0


Want to focus on lenders that receive 100 or more applications, so use lei (lender unique id) to remove anything less than 100 applications
Loan purpose, only want 1, home purchase
action-type: 1,2 application approved, 3 denied, ignore rest



In [3]:
# data clean up
# # want lei with greater than 100 overall applications
mortgage_df = mortgage_df.groupby('lei').filter(lambda x: len(x) >= 100)
# loan purpose: home purchase
mortgage_df = mortgage_df[mortgage_df['loan_purpose'] == 1]
# action taken 1,2 application approved 3 denied
mortgage_df = mortgage_df[mortgage_df['action_taken'].isin([1, 2, 3])]
# edit debt_to_income_ratio column to be numeric and replace exempt with NaN
dti_map = { "<20%": 10, "20%-<30%": 25, "30%-<36%": 33, "50%-60%": 55 , ">60%": 70 }
mortgage_df['debt_to_income_ratio'] = mortgage_df['debt_to_income_ratio'].replace(dti_map)
mortgage_df['debt_to_income_ratio'] = pd.to_numeric(mortgage_df['debt_to_income_ratio'], errors='coerce')


# for approval need action taken where values are 1 or 2
mortgage_df['approved'] = mortgage_df['action_taken'].apply(lambda x: 1 if x in [1,2] else 0)
# drop rows with missing values in debt_to_income_ratio and approved columns
mortgage_df = mortgage_df.dropna(subset=['debt_to_income_ratio'])


In [4]:
# get feature columns
features = ['loan_type', 'loan_amount', 'loan_to_value_ratio', 'debt_to_income_ratio', 'property_value', 'income']
x = mortgage_df[features]
y = mortgage_df['approved']

x = x.dropna()
y = y.loc[x.index]

print(x.shape)
print(y.shape)

(14748, 6)
(14748,)


In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

model = LogisticRegression(max_iter=1000)
model.fit(x_train_scaled, y_train)

y_pred = model.predict(x_test)

c:\Users\justi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[  51  269]
 [ 363 2267]]


In [7]:
coef_df = pd.DataFrame({
    'Feature': x.columns,
    'Coefficient': model.coef_[0]
})

coef_df.sort_values(
    by='Coefficient',
    ascending=False,
    inplace=True
)

print(coef_df)

                Feature  Coefficient
1           loan_amount     0.854111
0             loan_type     0.353070
5                income    -0.082976
4        property_value    -0.470887
3  debt_to_income_ratio    -0.835361
2   loan_to_value_ratio    -2.976731
